# Preprocessing

Le but de ce notebook est de travailler la donnée issue des verbatims pour pouvoir mener une analyse sémantique la plus riche possible.

Nous passerons donc :
- Par la création de colonnes qui regroupent certaines notes en catégories (3 )
- Par l'identification de signaux dans les verbatims (points d'exclamations et mots écrits en lettre CAPITALES)
- Par le nettoyage des mots inutiles
- Par la lemminisation, soit le regroupements des mots qui ont une même racine (ex : trouver, trouve --> 1 seul mot).

Dans un autre notebook, nous pourrons mener l'analyse sémantique (NLP), en essayant le BagOfWord, le TD-IDF, le N-series et ensuite le LDA pour regarder quelles mots / thématiques ressortent le plus par série et par niveau de satisfaction.

Par simplicité, nous ne garderons que les verbatims de Imdb, car en langue anglaise, et parce qu'il y en a nettement plus.

In [1]:
# Import des libraires
import pandas as pd, numpy as np, re

In [2]:
# Import du dataframe
df = pd.read_csv("df_clean.csv")

# Sélection du dataframe sur la plateforme IMDB pour simplifier le travail.
df = df[df["plateforme"] == 'Imdb']
df

,avis,dates,series,notes,plateforme
195,Honest Opinion - Turns out Westeros is way mor...,2026-01-24,A Knight of the Seven Kingdoms,8.0,Imdb
196,"So Far, So Good (Don't Screw It Up) - The firs...",2026-01-19,A Knight of the Seven Kingdoms,9.0,Imdb
197,A proper return to Westeros - I stayed up for ...,2026-01-19,A Knight of the Seven Kingdoms,9.0,Imdb
198,The only disappointment i had with ep 1 - Its ...,2026-01-19,A Knight of the Seven Kingdoms,9.0,Imdb
199,Really freaking good with only one bad moment ...,2026-01-19,A Knight of the Seven Kingdoms,9.0,Imdb
...,...,...,...,...,...
1602,Mediocre filmmaking - Weak script in combinati...,2025-10-29,King & Conqueror,1.0,Imdb
1603,drunk history - As many other reviewers have n...,2026-02-02,King & Conqueror,3.0,Imdb
1604,I get it... -,2026-02-26,King & Conqueror,7.0,Imdb
1605,Realism... - Earl's military crews count 5-8 s...,2026-03-02,King & Conqueror,3.0,Imdb


## 1. Ajout de colonnes

Enregistrement des informations les plus importantes : le regroupement de notes en 3 catégories et les métadonnées

In [24]:
# Création des fonctions pour créer des regroupements de la note (en 3 catégories)
def notes_3_cat(value):
    if value < 4:
        return -1
    elif 4 <= value < 7:
        return 0
    else:
        return 1

# Création des colonnes
df['notes_3'] = df['notes'].map(notes_3_cat)

In [25]:
# Vérification des effectifs de notes par série
for serie in df['series'].unique():
    print(f'Distribution des notes de la série {serie} : \n', df[df['series'] == serie]['notes_3'].value_counts())
    print()

# Des échantillons suffisants pour les catégories de notes des deux séries.

Distribution des notes de la série A Knight of the Seven Kingdoms : 
 notes_3
 1    1017
-1     127
 0      95
Name: count, dtype: int64

Distribution des notes de la série King & Conqueror : 
 notes_3
-1    81
 1    50
 0    42
Name: count, dtype: int64



In [ ]:
# Création des fonctions
def capital_count(text : str):
    '''Compter le nombre de mots écrits en lettres capitales dans un texte'''
    capital = re.compile(r'[A-Z]{2,}')
    capital_search = capital.findall(text)
    return len(capital_search)

def exclamation_count(text : str):
    '''Compter le nombre de points d'exclamation dans un texte'''
    exclamation = re.compile(r'!')
    exclamation_search = exclamation.findall(text)
    return len(exclamation_search)

def exclamation_chain_count(text : str):
    '''Compter le nombre de chaînes de points d'exclamation (!!!) dans un texte'''
    exclamation_chain = re.compile(r'[!]{2,}')
    exclamation_chain_search = exclamation_chain.findall(text)
    return len(exclamation_chain_search)

def interrogation_count(text : str):
    '''Compter le nombre de points d'interrogation dans un texte'''
    interrogation = re.compile(r'\?')
    interrogation_search = interrogation.findall(text)
    return len(interrogation_search)

def interrogation_chain_count(text : str):
    '''Compter le nombre de chaînes de points d'exclamation (!!!) dans un texte'''
    interrogation_chain = re.compile(r'[?]{2,}')
    interrogation_chain_search = interrogation_chain.findall(text)
    return len(interrogation_chain_search)

# Création des colonnes de métadonnées
df['nb_capital_words'] = df['avis'].apply(capital_count)
df['nb_exclamation_points'] = df['avis'].apply(exclamation_count)
df['nb_exclamation_points_chains'] = df['avis'].apply(exclamation_chain_count)
df['nb_interrogation_points'] = df['avis'].apply(interrogation_count)
df['nb_interrogation_points_chains'] = df['avis'].apply(interrogation_chain_count)
df['lenght'] = df['avis'].apply(len)

In [40]:
df.describe()

,notes,notes_3,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains,lenght
count,1412.000000,1412.000000,1412.000000,1412.000000,1412.000000,1412.000000,1412.000000,1412.000000
mean,7.956799,0.608357,0.952550,0.453258,0.051700,0.218130,0.012040,588.647309
std,3.010537,0.730242,3.374543,1.408669,0.270477,0.774695,0.132563,550.852360
min,1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000
25%,7.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,339.000000
50%,10.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,419.000000
75%,10.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,684.000000
max,10.000000,1.000000,67.000000,27.000000,3.000000,12.000000,2.000000,4995.000000


In [41]:
# Regardons directement les commentaires qui ont des métadonnées supérieures à 1.
# Nous traiterons le cas de 'length' différemment

df_metadonnees = df[['nb_capital_words', 'nb_exclamation_points', 'nb_exclamation_points_chains', \
                     'nb_interrogation_points', 'nb_interrogation_points_chains']]

for col in df_metadonnees:
    print(col, ' :')
    display(df[['avis', 'notes_3']][df_metadonnees[col] >= 1])
    print() # Saut de ligne

nb_capital_words  :


,avis,notes_3
196,"So Far, So Good (Don't Screw It Up) - The firs...",1
197,A proper return to Westeros - I stayed up for ...,1
199,Really freaking good with only one bad moment ...,1
202,Dunk is no lunk! - Wow! Just WOW! A Knight of ...,1
203,Short episodes but definitely enjoyable - Over...,1
...,...,...
1588,Utter nonsense - By all means make a drama set...,-1
1593,GOT fictional insult to English history - BBC ...,-1
1598,Well..... - I'm certain I'm writing a review m...,0
1599,Disapointment - I Had high hopes. It is such a...,-1



nb_exclamation_points  :


,avis,notes_3
198,The only disappointment i had with ep 1 - Its ...,1
202,Dunk is no lunk! - Wow! Just WOW! A Knight of ...,1
206,Great - I'm absolutely loving A Knight of the ...,1
208,"Epic, love this - Episode 2 is only 30 minutes...",1
213,"Great start! - Very good pilot, the cinematogr...",1
...,...,...
1586,Worse than bad!! - I cannot begin to try to ex...,-1
1589,Teenage drama trying to be serious... - Was Ha...,0
1595,Soap Opera Video Quality so I didn't even watc...,-1
1599,Disapointment - I Had high hopes. It is such a...,-1



nb_exclamation_points_chains  :


,avis,notes_3
294,Loved the Cinematography!!! - Absolute cinemat...,1
298,Best Casting! - Dunk & Egg's Casting is Amazin...,1
305,Series might just be a another GOT and i bet i...,1
389,HBO/Martin's next strike!!! - Having seen the ...,1
426,LOVEEED - Love love love (Mister finn bennett ...,1
434,Almost perfect - I like that the series doesn'...,1
443,Dunk and Egg!!!! - What a fantastic series til...,1
457,"Finally, Someone to Root For!! - A Knight of t...",1
470,Out Freakin Standing!!! - This has the potenti...,1
516,Surprising! - Not one to leave reviews but I w...,1



nb_interrogation_points  :


,avis,notes_3
198,The only disappointment i had with ep 1 - Its ...,1
211,Landmark TV - Episode 1 Jan 19. 2026\n\nSimple...,1
215,2 good episodes out of 6 - With such rich Iris...,0
217,"Fantasy From the Road, Not the Throne - This r...",1
218,A small and perfect GOT story - From the momen...,1
...,...,...
1589,Teenage drama trying to be serious... - Was Ha...,0
1595,Soap Opera Video Quality so I didn't even watc...,-1
1596,Why the mini horses? - I had trouble taking th...,0
1602,Mediocre filmmaking - Weak script in combinati...,-1



nb_interrogation_points_chains  :


,avis,notes_3
283,Why only 40 mins and one episode??? - The firs...,1
348,"Very boring snoozefest, perhaps suitable for 8...",-1
356,"Terrible start, but improving - I was looking ...",1
431,Got level goated ?? - Episode 1 begins on a ve...,1
460,FREAKING AWESOME! - Unlike all the other dummi...,1
569,Charming but not Game of Thrones - It's defini...,0
726,Boooooring - What is up with those reviews??? ...,-1
849,I see emmy sphree - Perfect casting...good act...,1
931,Very good GoT prequel but short episodes/seaso...,1
1117,10/10!!!!! - Well what can one say??? I guess ...,1


In [ ]:
# Essayer avec des grpahiques, des boîtes à moustache.

,nb_capital_words,nb_exclamation_points,nb_exclamation_points_chains,nb_interrogation_points,nb_interrogation_points_chains
notes_3,,,,,
-1,1.076923,0.331731,0.024038,0.620192,0.033654
0,1.065693,0.270073,0.007299,0.503650,0.007299
1,0.913777,0.500469,0.062793,0.103093,0.008435
